In [ ]:
#q1
import pandas as pd

roll_number = '1024160130'
print(f"Using roll number: {roll_number}")

# Fixed FAQ entries provided in the question
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# Extract last two digits of the roll number
last_two_digits_str = roll_number[-2:]
digit1 = int(last_two_digits_str[0])
digit2 = int(last_two_digits_str[1])

print(f"Last two digits: {digit1}, {digit2}")

categories = ["billing", "account", "general"]

# Create personalized entries
personalized_entries = []

# Entry 1 based on digit1
category1 = categories[digit1 % 3]
if category1 == 'billing':
    q1 = "What are the charges for late payment?"
    a1 = "A late payment fee of Rs 100 will be applied."
    k1 = "late payment charges fee penalty"
elif category1 == 'account':
    q1 = "How do I change my registered email address?"
    a1 = "You can update your email in the 'Profile Settings' section."
    k1 = "email change update profile"
else: # general
    q1 = "Where can I find the nearest branch?"
    a1 = "Our branch locator is available on the website's 'Contact Us' page."
    k1 = "branch location address map"
personalized_entries.append({"question": q1, "answer": a1, "keywords": k1, "category": category1})

# Entry 2 based on digit2
category2 = categories[digit2 % 3]
if category2 == 'billing':
    q2 = "Is there a refund policy?"
    a2 = "Yes, refunds are processed within 7-10 business days."
    k2 = "refund policy money back cancel"
elif category2 == 'account':
    q2 = "Can I link multiple accounts?"
    a2 = "No, only one account can be linked per user profile."
    k2 = "link accounts multiple profiles"
else: # general
    q2 = "What are your privacy policies?"
    a2 = "Our privacy policy details how we handle your data, available on our legal page."
    k2 = "privacy data policy security"
personalized_entries.append({"question": q2, "answer": a2, "keywords": k2, "category": category2})

# Combine fixed and personalized entries
df = pd.DataFrame(fixed_entries + personalized_entries)

# Print the final 6-row DataFrame
print("\nFinal 6-row DataFrame:")
display(df)

In [ ]:
#q2
import re

def score_query(query, df):
    """
    Scores FAQ entries based on how many keywords from the entry match words in the query.
    Returns a list of dictionaries, each containing the entry's details and its confidence score,
    sorted by confidence in descending order.
    """
    query_words = set(re.findall(r'\b\w+\b', query.lower())) # Extract and lowercase words from query

    scored_entries = []
    for index, row in df.iterrows():
        entry_keywords = set(re.findall(r'\b\w+\b', row['keywords'].lower())) # Extract and lowercase keywords from entry

        # Calculate confidence as the number of matching keywords
        confidence = len(query_words.intersection(entry_keywords))

        scored_entries.append({
            "question": row['question'],
            "answer": row['answer'],
            "category": row['category'],
            "confidence": confidence
        })

    # Sort by confidence in descending order
    scored_entries.sort(key=lambda x: x['confidence'], reverse=True)
    return scored_entries


# Demonstrate the scoring function with a sample query
print("\nDemonstrating scoring function for query: 'how can I reset my password?'")
query = "how can I reset my password?"
ranked_faqs = score_query(query, df)

for entry in ranked_faqs:
    print(f"\nQuestion: {entry['question']}")
    print(f"Answer: {entry['answer']}")
    print(f"Category: {entry['category']}")
    print(f"Confidence: {entry['confidence']}")

In [ ]:
#q3
def same_category(category_name, df):

    return df[df['category'].str.lower() == category_name.lower()]



# Let's dynamically pick one of the personalized categories from the df
# For example, the category of the 5th entry (index 4) which is one of the personalized ones.
if len(df) > 4:
    sample_category = df.loc[4, 'category']
else:
    # Fallback if for some reason personalized entries aren't at expected indices
    sample_category = 'account' # Default to a common category

print(f"\nDemonstrating same_category for category: '{sample_category}'")
category_faqs = same_category(sample_category, df)
display(category_faqs[['question', 'category']])

In [ ]:
#q4
import os

entry_index = 0
print(f"Original keywords for entry {entry_index}: {df.loc[entry_index, 'keywords']}")

# Ask the user to input a new keyword (using a placeholder for non-interactive environment)
# Please change 'new_important_keyword' to your desired keyword.
new_keyword = 'new_important_keyword'

# Add the new keyword to that entry's keywords
# Ensure no leading/trailing spaces for cleaner comma separation
current_keywords = df.loc[entry_index, 'keywords'].strip()
if current_keywords:
    df.loc[entry_index, 'keywords'] = f"{current_keywords}, {new_keyword.strip()}"
else:
    df.loc[entry_index, 'keywords'] = new_keyword.strip()

print(f"Updated keywords for entry {entry_index}: {df.loc[entry_index, 'keywords']}")

# Save the entire updated DataFrame to a CSV file
csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)

print(f"\nDataFrame saved to '{csv_filename}'")
print("Content of the CSV file:")
with open(csv_filename, 'r') as f:
    print(f.read())

# Display the updated DataFrame to verify
print("\nUpdated DataFrame with new keyword:")
display(df)

In [ ]:
#q5
# Using groupby, count how many FAQ entries you have per category
category_counts = df.groupby('category').size().reset_index(name='count')
print("\nFAQ entries per category:")
display(category_counts)

In [ ]:
#q6
def score_query_with_ties(query, df):
    """
    Scores FAQ entries, handling ties by returning all entries with the highest score.
    """
    query_words = set(re.findall(r'\b\w+\b', query.lower()))

    scored_entries = []
    for index, row in df.iterrows():
        entry_keywords = set(re.findall(r'\b\w+\b', row['keywords'].lower()))
        confidence = len(query_words.intersection(entry_keywords))
        scored_entries.append({
            "question": row['question'],
            "answer": row['answer'],
            "category": row['category'],
            "confidence": confidence
        })

    if not scored_entries:
        return []

    # Find the maximum confidence score
    max_confidence = max(entry['confidence'] for entry in scored_entries)

    # Filter for entries that have the maximum confidence score
    tied_entries = [entry for entry in scored_entries if entry['confidence'] == max_confidence and max_confidence > 0]

    # Sort tied entries (optional, but good for consistent output)
    tied_entries.sort(key=lambda x: x['question'])

    return tied_entries

# Demonstrate with a query that produces a tie (e.g., 'fee' related queries)
print("\n--- Demonstrating query with a tie (e.g., 'payment fee') ---")
tie_query = "I need to know about the payment fee"
tied_results = score_query_with_ties(tie_query, df)

if tied_results:
    print(f"Query: '{tie_query}'")
    for entry in tied_results:
        print(f"\nQuestion: {entry['question']}")
        print(f"Answer: {entry['answer']}")
        print(f"Category: {entry['category']}")
        print(f"Confidence: {entry['confidence']}")
else:
    print("No matching entries found for tie-producing query.")

# Demonstrate with a query that doesn't produce a tie
print("\n--- Demonstrating query without a tie (e.g., 'working hours') ---")
no_tie_query = "What are your working hours?"
no_tie_results = score_query_with_ties(no_tie_query, df)

if no_tie_results:
    print(f"Query: '{no_tie_query}'")
    for entry in no_tie_results:
        print(f"\nQuestion: {entry['question']}")
        print(f"Answer: {entry['answer']}")
        print(f"Category: {entry['category']}")
        print(f"Confidence: {entry['confidence']}")
else:
    print("No matching entries found for non-tie-producing query.")